In [2]:
import torch
import time
import pandas as pd
import numpy as np
from timm import create_model
import random

print("🔥 Imports Loaded!!!")

🔥 Imports Loaded!!!


In [4]:
# ---------------------------
# CONFIG
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device", device)
input_size = 224
batch_size = 1
num_warmup = 30
num_iters = 100
seeds = [0, 1, 2, 3, 4]   # ✅ multiple seeds

# ---------------------------
# MODEL LIST
# ---------------------------
models = {
    "EfficientFormerV2-S0": "efficientformerv2_s0",
    "EdgeNeXt-XX-S": "edgenext_xx_small",
    "FastViT-T8": "fastvit_t8",
    "EfficientFormer-L1": "efficientformer_l1",
    "TinyViT-5M": "tiny_vit_5m_224",
    "MobileNetV4-Small": "mobilenetv4_conv_small.e2400_r224_in1k",
    "MobileOne-S1": "mobileone_s1",
    "TinyNet-E": "tinynet_e",
    "EfficientViT-M0": "efficientvit_m0",
    "ViT-Small DINOv3": "vit_small_patch16_224"
}

# ---------------------------
# HELPERS
# ---------------------------
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def measure_latency(model, input_tensor):
    model.eval()

    # warmup
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(input_tensor)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.time()
    with torch.no_grad():
        for _ in range(num_iters):
            _ = model(input_tensor)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    end = time.time()

    latency = (end - start) / num_iters * 1000  # ms
    fps = 1000 / latency
    return latency, fps

# ---------------------------
# MAIN LOOP
# ---------------------------
results = []

for name, model_name in models.items():
    print(f"\nRunning model: {name}")

    lat_list = []
    fps_list = []

    model = create_model(model_name, pretrained=False).to(device)
    params = count_params(model)

    for seed in seeds:
        set_seed(seed)
        input_tensor = torch.randn(batch_size, 3, input_size, input_size).to(device)

        latency, fps = measure_latency(model, input_tensor)

        lat_list.append(latency)
        fps_list.append(fps)

    # ✅ Mean ± Std
    lat_mean = np.mean(lat_list)
    lat_std = np.std(lat_list)

    fps_mean = np.mean(fps_list)
    fps_std = np.std(fps_list)

    results.append({
        "Model": name,
        "Params (M)": round(params, 2),
        "Latency (ms)": f"{lat_mean:.2f} ± {lat_std:.2f}",
        "FPS": f"{fps_mean:.2f} ± {fps_std:.2f}"
    })

# ---------------------------
# OUTPUT
# ---------------------------
df = pd.DataFrame(results)
print("\nFinal Results:")
print(df)

df.to_csv("benchmark_multiseed.csv", index=False)

Using Device cpu

Running model: EfficientFormerV2-S0

Running model: EdgeNeXt-XX-S

Running model: FastViT-T8

Running model: EfficientFormer-L1

Running model: TinyViT-5M

Running model: MobileNetV4-Small

Running model: MobileOne-S1

Running model: TinyNet-E

Running model: EfficientViT-M0

Running model: ViT-Small DINOv3

Final Results:
                  Model  Params (M)  Latency (ms)           FPS
0  EfficientFormerV2-S0        3.60  36.73 ± 0.27  27.23 ± 0.20
1         EdgeNeXt-XX-S        1.33  15.42 ± 0.25  64.88 ± 1.05
2            FastViT-T8        4.03  41.41 ± 0.48  24.15 ± 0.28
3    EfficientFormer-L1       12.29  55.13 ± 0.20  18.14 ± 0.06
4            TinyViT-5M       12.08  56.94 ± 0.42  17.56 ± 0.13
5     MobileNetV4-Small        3.77  14.87 ± 0.07  67.27 ± 0.31
6          MobileOne-S1        4.83  45.02 ± 0.47  22.21 ± 0.23
7             TinyNet-E        2.04  14.74 ± 0.18  67.85 ± 0.82
8       EfficientViT-M0        2.35  24.32 ± 0.28  41.13 ± 0.48
9      ViT-Small 